In [9]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("wfp_food_prices_nga.csv")

print(df.head())

         date   admin1 admin2       market  market_id  latitude  longitude  \
0  2002-01-15  Katsina  Jibia  Jibia (CBM)       1038     13.08       7.24   
1  2002-01-15  Katsina  Jibia  Jibia (CBM)       1038     13.08       7.24   
2  2002-01-15  Katsina  Jibia  Jibia (CBM)       1038     13.08       7.24   
3  2002-01-15  Katsina  Jibia  Jibia (CBM)       1038     13.08       7.24   
4  2002-01-15  Katsina  Jibia  Jibia (CBM)       1038     13.08       7.24   

             category        commodity  commodity_id unit priceflag  \
0  cereals and tubers            Maize            51   KG    actual   
1  cereals and tubers           Millet            73   KG    actual   
2  cereals and tubers  Rice (imported)            64   KG    actual   
3  cereals and tubers          Sorghum            65   KG    actual   
4     pulses and nuts    Beans (niebe)           120   KG    actual   

   pricetype currency   price  usdprice  
0  Wholesale      NGN  175.92      1.54  
1  Wholesale      NG

In [10]:

print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.isnull().sum())
print(df.describe())

(91216, 16)
Index(['date', 'admin1', 'admin2', 'market', 'market_id', 'latitude',
       'longitude', 'category', 'commodity', 'commodity_id', 'unit',
       'priceflag', 'pricetype', 'currency', 'price', 'usdprice'],
      dtype='object')
date             object
admin1           object
admin2           object
market           object
market_id         int64
latitude        float64
longitude       float64
category         object
commodity        object
commodity_id      int64
unit             object
priceflag        object
pricetype        object
currency         object
price           float64
usdprice        float64
dtype: object
date            0
admin1          0
admin2          0
market          0
market_id       0
latitude        0
longitude       0
category        0
commodity       0
commodity_id    0
unit            0
priceflag       0
pricetype       0
currency        0
price           0
usdprice        0
dtype: int64
          market_id      latitude     longitude  commodity_id

In [11]:
# converting date to datetime
df["date"] = pd.to_datetime(df["date"])

# selecting 2021 dates
df = df[df["date"] >= "2021-01-01"]

# counting records by state
print(df["admin1"].value_counts())

# by market
print(df["market"].value_counts().head(20))

#by commodity
print(df["commodity"].value_counts().head(20))

admin1
Borno      19705
Yobe       15551
Adamawa     1535
Kaduna      1362
Gombe        750
Kebbi        750
Zamfara      750
Oyo          749
Kano         728
Lagos        725
Abia         690
Katsina      684
Jigawa       650
Name: count, dtype: int64
market
Potiskum             2067
Gujba (Buni Yadi)    1507
Bade (Gashua)        1371
Monday               1333
Custom               1285
Abba Gamaram         1283
Budum                1249
Geidam               1212
Tashan Bama          1193
Yusufari             1183
Yunusari             1176
Baga Road            1170
Bayantasha            923
Gulani (Tettaba)      920
Damassack (CBM)       882
Monguno               879
Mafa                  874
Bullunkutu            869
Damaturu              825
Damagum               819
Name: count, dtype: int64
commodity
Rice (local)       2121
Millet             2108
Yam                2025
Rice (imported)    1798
Oil (palm)         1728
Onions             1346
Sugar              1346
Beans (white)  

In [12]:
# removing duplicate rows
df = df.drop_duplicates()

# removing rows without important information
df = df.dropna(subset=["date", "market", "commodity", "price"])

# keep naira prices
df = df[df["currency"] == "NGN"]

# retail prices
df = df[df["pricetype"] == "Retail"]

# prices measured in kg
df = df[df["unit"] == "KG"]

# Remove zero or negative prices
df = df[df["price"] > 0]

print(df.head())
print("Remaining rows:", len(df))

            date admin1   admin2        market  market_id  latitude  \
41921 2021-01-15  Borno  Konduga  Abba Gamaram       2394     11.86   
41922 2021-01-15  Borno  Konduga  Abba Gamaram       2394     11.86   
41923 2021-01-15  Borno  Konduga  Abba Gamaram       2394     11.86   
41945 2021-01-15  Borno  Konduga     Baga Road       2385     11.87   
41946 2021-01-15  Borno  Konduga     Baga Road       2385     11.87   

       longitude             category    commodity  commodity_id unit  \
41921      13.15  meat, fish and eggs         Fish           139   KG   
41922      13.15  meat, fish and eggs  Meat (beef)           141   KG   
41923      13.15  meat, fish and eggs  Meat (goat)           451   KG   
41945      13.12  meat, fish and eggs         Fish           139   KG   
41946      13.12  meat, fish and eggs  Meat (beef)           141   KG   

      priceflag pricetype currency   price  usdprice  
41921    actual    Retail      NGN   524.0      1.38  
41922    actual    Retai

In [13]:
commodities = ["Rice (local)","Maize (white)","Beans (white)","Gari (white)","Yam","Tomatoes"]

df = df[df["commodity"].isin(commodities)]

In [14]:
# Extract month and year
df["date"] = df["date"].dt.to_period("M")

# calculating monthly average price
monthly_data = df.groupby(
    ["date", "admin1", "market", "commodity"]
)["price"].mean().reset_index()

print(monthly_data.head())

      date   admin1     market      commodity   price
0  2021-01  Adamawa       Mubi   Gari (white)  323.38
1  2021-01  Adamawa       Mubi  Maize (white)  151.84
2  2021-01    Borno        Biu   Gari (white)  332.23
3  2021-01    Borno        Biu  Maize (white)  158.79
4  2021-01    Borno  Maiduguri   Gari (white)  348.26


In [15]:
# missing values
print(monthly_data.isnull().sum())

#average price of each commodity
print(monthly_data.groupby("commodity")["price"].mean())

# average price by market
print(monthly_data.groupby("market")["price"].mean())

#highest food prices
print(monthly_data.sort_values("price", ascending=False).head(10))

date         0
admin1       0
market       0
commodity    0
price        0
dtype: int64
commodity
Gari (white)     342.155174
Maize (white)    214.308970
Name: price, dtype: float64
market
Aba             345.833913
Biu             175.893600
Damaturu        330.030400
Dandume         194.539200
Dawanau         255.171600
Giwa            248.331600
Gombe           336.241800
Gujungu         260.177400
Gwandu          243.100200
Ibadan          244.409000
Kaura Namoda    280.830200
Lagos           333.807600
Maiduguri       309.197000
Mubi            258.288000
Potiskum        276.206400
Saminaka        291.255400
Name: price, dtype: float64
        date  admin1    market     commodity   price
203  2021-07   Lagos     Lagos  Gari (white)  568.79
610  2022-08    Yobe  Damaturu  Gari (white)  550.07
579  2022-07    Yobe  Damaturu  Gari (white)  545.51
227  2021-08  Kaduna  Saminaka  Gari (white)  540.64
258  2021-09  Kaduna  Saminaka  Gari (white)  539.76
486  2022-04    Yobe  Damaturu  G

In [16]:
monthly_data.to_csv("cleaned_food_prices.csv", index=False)
